# Gnasher Group — Version 2: trained seven-group classifier
**Irene Ramos · ITAI 1378 · Tier 1**

This is the next development stage, not a completed training report. No V2
accuracy, speed, confusion matrix, or trained weights have been recorded yet.
Version 1 is preserved under `prototypes/v1/` with its original saved outputs.

**What you will learn:** labeling, data splits, transfer learning, validation,
and paired testing. We train a new seven-output layer, then fine-tune the last
three feature blocks of pretrained EfficientNet-B0. The existing model is our
baseline, not discarded work.

Before running, choose **Runtime → Change runtime type → T4 GPU** in Colab.
Keep this notebook in your Google Drive. Training requires you to review the
breed table first. Run cells step by step rather than blindly running all.

## 1. Set up the repository and packages
The repository is public. No GitHub token is needed. The cell records the actual
source commit used so an experiment can be traced to its code.

In [ ]:
%pip install -q torch torchvision pillow matplotlib pandas numpy
from pathlib import Path
import subprocess, sys, json

REPO = Path('/content/gnasher-final-source')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/IreneAlexRamos/itai1378-project.git',str(REPO)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
sys.path.insert(0,str(REPO / 'src'))
sys.path.insert(0,str(REPO))
print('Source commit:',SOURCE_COMMIT)

## 2. Save artifacts in Drive and set the experiment
The source images live in the temporary Colab runtime for faster access.
Reviewed labels, splits, checkpoints, and reports go to Drive. Use a new
experiment name when changing labels, settings, or training again. A disconnected
run keeps its `best.pt`, `last.pt`, and history; these can be loaded for inference.
The initial workflow does not resume an interrupted optimizer state.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, torchvision
from gnasher_data import download_dataset, draft_label_table, read_reviewed_labels, build_manifest, GROUPS
from gnasher_training import make_loaders, train_model, load_best, select_threshold, evaluate_paired, benchmark_model, predict_photo
from gnasher_group_mapping import BREED_TO_GROUP, UNSUPPORTED_DOG_LABELS, aggregate_group_scores

PROJECT = Path('/content/drive/MyDrive/GnasherGroup')
EXPERIMENT_NAME = 'v2_run01'  # Change for a genuinely new experiment.
EXPERIMENT = PROJECT / 'experiments' / EXPERIMENT_NAME
EXPERIMENT.mkdir(parents=True,exist_ok=True)
REVIEW_TABLE = PROJECT / 'stanford_breed_to_group_review.csv'
DATA_ROOT = Path('/content/stanford_dogs')
SEED, CAP_PER_GROUP, BATCH_SIZE = 42, 1000, 32
HEAD_EPOCHS, FINE_EPOCHS = 3, 5
TRAIN_NOW = True  # Set False to load an existing best.pt without retraining.
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU selected')
print('Artifacts:',EXPERIMENT)

## 3. Get the original Stanford Dogs photographs
Source: http://vision.stanford.edu/aditya86/ImageNetDogs/

The original dataset has 20,580 images and 120 source breed folders. The download
may take time. If Stanford is unavailable, obtain `images.tar` from the official
page and upload it to `/content/stanford_dogs/`, then rerun this cell.

We use full photographs, not bounding-box crops. The model's validation/test
preprocessing resizes and center-crops them. **Stanford Dogs is ImageNet-derived**;
test separation below is from this project's fine-tuning, not proof of separation
from ImageNet pretraining.

In [ ]:
IMAGES = download_dataset(DATA_ROOT)
print('Image folders:',len([p for p in IMAGES.iterdir() if p.is_dir()]))
print('Photographs:',len(list(IMAGES.glob('*/*.jpg'))))

## 4. Make It Yours — review the actual breed labels
This is real labeling work. The helper inventories the downloaded dataset's
exact folder identifiers and suggests labels from V1; it does **not** certify them.

1. Open the CSV in a spreadsheet. Review each canonical breed and AKC group using
   an official AKC breed page; correct aliases, varieties, and source URLs.
2. Keep unresolved labels excluded (`include=no`) and explain why in `notes`.
   Do not assume “Eskimo dog” means American Eskimo Dog. Wild-canid or unsupported
   categories also need documented exclusions rather than a forced group.
3. After actually checking each row, set `review_status=reviewed` and
   `reviewed_by=Irene Ramos`. Keep all source identifiers and all columns.
4. Export the reviewed spreadsheet as CSV and upload it in the next cell.

Do not mark rows reviewed just to bypass the check. This work should appear in
your reflection and AI usage log. A cap of 1,000 per group is a maximum;
exclusions or availability can make the total less than 7,000.

In [ ]:
import pandas as pd
draft_label_table(IMAGES,BREED_TO_GROUP,UNSUPPORTED_DOG_LABELS,REVIEW_TABLE)
labels = pd.read_csv(REVIEW_TABLE,keep_default_na=False)
display(labels)
print('CSV to review:',REVIEW_TABLE)
# Optional download for editing in a spreadsheet:
# from google.colab import files
# files.download(str(REVIEW_TABLE))

In [ ]:
IMPORT_REVIEWED_CSV = False  # Set True when your reviewed CSV is ready to upload.
if IMPORT_REVIEWED_CSV:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
    if len(csv_files) != 1:
        raise ValueError('Upload exactly one reviewed breed-table CSV.')
    # Validate the upload before replacing the persisted table.
    candidate = PROJECT / 'candidate_review.csv'
    candidate.write_bytes(uploaded[csv_files[0]])
    read_reviewed_labels(candidate,IMAGES)
    candidate.replace(REVIEW_TABLE)
reviewed = read_reviewed_labels(REVIEW_TABLE,IMAGES)
print('Reviewed rows:',len(reviewed))

## 5. Create a fixed 70/15/15 split
The manifest labels every selected photograph with its source breed, group,
split, and SHA-256. Exact-byte duplicates are removed before sampling. A
conflicting duplicate label stops the workflow. Near duplicates remain a
limitation. The custom split follows the proposal; it differs from the dataset's
official 12,000/8,580 split and is not comparable to that benchmark directly.

Only training gets random image augmentation. Validation and test use the
pretrained model's deterministic transforms. The test set is not used for
training, model selection, or threshold selection.

In [ ]:
manifest = build_manifest(IMAGES,REVIEW_TABLE,EXPERIMENT,cap=CAP_PER_GROUP,seed=SEED)
metadata = json.loads((EXPERIMENT/'manifest_metadata.json').read_text())
display(pd.DataFrame(manifest).groupby(['group','split']).size().unstack(fill_value=0).reindex(GROUPS))
print('Selected photographs:',len(manifest))
run_config = {'source_commit':SOURCE_COMMIT,'seed':SEED,'cap_per_group':CAP_PER_GROUP,
              'batch_size':BATCH_SIZE,'head_epochs':HEAD_EPOCHS,'fine_epochs':FINE_EPOCHS,
              'torch':str(torch.__version__),'torchvision':str(torchvision.__version__),
              'manifest_sha256':metadata['manifest_sha256']}
config_path = EXPERIMENT/'run_config.json'
if config_path.exists() and json.loads(config_path.read_text()) != run_config:
    raise ValueError('Experiment configuration changed. Use a new experiment name.')
config_path.write_text(json.dumps(run_config,indent=2))
loaders = make_loaders(IMAGES,manifest,batch_size=BATCH_SIZE,seed=SEED)

## 6. Train the seven-output head, then fine-tune
V1 summed outputs from a pretrained 1,000-label model. V2 replaces the final
layer with seven outputs and learns from the reviewed group labels.

First the feature extractor is frozen while the new head trains. Then the last
three feature blocks are unfrozen with a smaller learning rate. Frozen layers'
BatchNorm statistics remain fixed. Class weights come from **training counts**.
Validation chooses the best checkpoint and early stopping; test is untouched.
Default epochs are a starting experiment, not a promise of 85% accuracy.

In [ ]:
if TRAIN_NOW:
    if not torch.cuda.is_available():
        raise RuntimeError('Select a Colab GPU before starting the full training run.')
    model, device = train_model(loaders,EXPERIMENT,metadata,seed=SEED,
                               head_epochs=HEAD_EPOCHS,fine_epochs=FINE_EPOCHS)
else:
    model, device, checkpoint = load_best(EXPERIMENT)
print('Checkpoint:',EXPERIMENT/'best.pt')

## 7. Inspect training and validation curves
Look for learning, unstable training, and a widening train/validation gap.
Describe the actual curves in your own words. Do not open test results to decide
which epoch or settings to keep.

In [ ]:
import matplotlib.pyplot as plt
history = pd.read_csv(EXPERIMENT/'history.csv')
display(history)
fig,axes = plt.subplots(1,2,figsize=(12,4))
x = range(1,len(history)+1)
for split in ['train','validation']:
    axes[0].plot(x,history[f'{split}_loss'],label=split)
    axes[1].plot(x,history[f'{split}_accuracy'],label=split)
axes[0].set_title('Loss'); axes[1].set_title('Accuracy')
for ax in axes:
    ax.set_xlabel('Epoch across both stages'); ax.legend()
fig.tight_layout(); fig.savefig(EXPERIMENT/'learning_curves.png',dpi=160); plt.show()

## 8. Lock the confidence threshold using validation
The provisional rule tries thresholds from 0.50 through 0.95. It picks the
smallest with at least 80% validation coverage and 85% accepted accuracy, falling
back to 0.50 when none qualify. This is a declared experiment rule, not
calibration or proof of non-dog detection. We will report full top-1 accuracy,
coverage, accepted accuracy, and abstentions separately.

In [ ]:
selection = select_threshold(model,loaders['validation'],device,EXPERIMENT)
display(pd.DataFrame(selection['candidates']))
print('Locked threshold:',selection['threshold'])
print('Validation coverage/accuracy criterion met:',selection['criterion_met'])

## 9. Evaluate once on the held-out test photographs
Compare **both versions on exactly the same images**. V1 keeps its 50% rule; V2
uses its validation-selected threshold. The old eight-photo demonstration rate
is not a fair comparison with this larger test split.

The primary classifier result is full top-1 test accuracy. The decision accuracy
keeps abstentions in the denominator. Accepted accuracy is a separate metric.
Do not change settings based on these test results and reuse the same test set
to claim an unbiased final improvement. Future retuning needs a new independent
evaluation or an explicitly disclosed exploratory analysis.

In [ ]:
report = evaluate_paired(model,loaders['test'],device,EXPERIMENT,aggregate_group_scores)
display(pd.DataFrame({name:{key:value for key,value in report[name].items()
                           if key not in ['confusion_matrix','decision_matrix_with_uncertain','per_group_accuracy']}
                      for name in ['prototype_v1','trained_v2']}))
print(report['limitations'])

In [ ]:
import numpy as np
matrix = np.array(report['trained_v2']['decision_matrix_with_uncertain'])
fig,ax = plt.subplots(figsize=(10,7))
im=ax.imshow(matrix,cmap='Blues')
ax.set_xticks(range(8),GROUPS+['Uncertain'],rotation=45,ha='right')
ax.set_yticks(range(7),GROUPS)
ax.set_xlabel('Final decision'); ax.set_ylabel('Expected group')
ax.set_title('Version 2 test decisions — abstentions retained')
for i in range(7):
    for j in range(8):
        ax.text(j,i,str(matrix[i,j]),ha='center',va='center')
fig.colorbar(im,ax=ax); fig.tight_layout()
fig.savefig(EXPERIMENT/'confusion_with_uncertain.png',dpi=160); plt.show()

## 10. Inspect 3–5 actual errors
Use the predicted file paths and scores, not made-up screenshots. Include
abstentions when discussing decision failures. If fewer than five errors exist,
report the actual number. This inspection is explanation after evaluation,
not permission to retune against the test set.

In [ ]:
from PIL import Image
predictions = pd.read_csv(EXPERIMENT/'test_predictions.csv')
errors = predictions[predictions['v2_decision']!=predictions['expected_group']].head(5)
display(errors[['relative_path','expected_group','v2_top1','v2_top1_score','v2_decision']])
if len(errors):
    fig,axes=plt.subplots(1,len(errors),figsize=(4*len(errors),4),squeeze=False)
    for ax,(_,row) in zip(axes[0],errors.iterrows()):
        with Image.open(IMAGES/row['relative_path']) as photo:
            ax.imshow(photo.convert('RGB'))
        ax.set_title(f"Expected: {row['expected_group']}\nDecision: {row['v2_decision']}")
        ax.axis('off')
    fig.tight_layout(); fig.savefig(EXPERIMENT/'failure_examples.png',dpi=160); plt.show()
else:
    print('No decision errors in this run; report that outcome without inventing errors.')

## 11. Measure speed with the actual device
This reproduces V1's forward-only timing scope. It does not measure a whole
application response. Compare CPU to CPU or GPU to GPU; the historic V1 CPU
number is not an apples-to-apples comparison with a new GPU number.

In [ ]:
speed = benchmark_model(model,IMAGES/loaders['test'].dataset.rows[0]['relative_path'],device)
(EXPERIMENT/'speed_report.json').write_text(json.dumps(speed,indent=2))
display(pd.DataFrame([speed]))

## 12. Try your own photograph and explain the limits
Upload Poppy, the husky, or another photograph. These are demonstrations, not
extra rows added to the held-out test score. You may also inspect Goober, but a
seven-group classifier has no cat class and can be confidently wrong on cats.
A low-confidence rule alone is not a validated non-dog detector.

In [ ]:
DEMO_IMAGE = Path('/content/bordercollieex.jpeg')  # Upload your photograph, then edit this path.
if DEMO_IMAGE.exists():
    decision = predict_photo(model,DEMO_IMAGE,device,selection['threshold'])
    display(Image.open(DEMO_IMAGE).convert('RGB'))
    print(json.dumps(decision,indent=2))
else:
    print('Optional demo: upload a photograph and edit DEMO_IMAGE, then rerun this cell.')

## 13. Your reflection and submission evidence
Replace the prompts below with your own observations after you run the work.

- Which labels or aliases did I verify/correct, and what sources helped?
- How did the head-training and fine-tuning stages differ in my curves?
- What are the paired V1/V2 test results? Did training actually improve them?
- What do my failure examples show? What does the model still not do?
- How did the real image counts and scope differ from my Blueprint?

Keep `stanford_breed_to_group_review.csv`, `manifest.csv`,
`manifest_metadata.json`, `run_config.json`, `history.csv`, `threshold.json`,
`test_predictions.csv`, `test_report.json`, `speed_report.json`, and the charts.
Keep `best.pt` in Drive; do not put dataset images or large weights in GitHub.
Save a notebook copy with actual outputs. Update the AI log truthfully and
revise the final slides only after real results exist.

## Sources and AI assistance
- Stanford Dogs: http://vision.stanford.edu/aditya86/ImageNetDogs/
- Dataset description: https://www.tensorflow.org/datasets/catalog/stanford_dogs
- AKC breed references: https://www.akc.org/dog-breeds/
- EfficientNet-B0: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b0.html
- Transfer learning: https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html

ChatGPT/Codex assisted with this workflow and starter training code. Irene's
label review, real training run, observations, and edits remain to be completed.
No human review or training results are being claimed by this blank notebook.